# Lab 6: Learning AR Models (Transformers hands-on)

This notebook covers the three topics of the hands-on session (`labs/transformers/Transformer.pdf`, `AI_lab_transformers.ipynb`, `Run_Ollama.ipynb`):

1. **Encoder-decoder** Transformers for machine translation (the original use case of the Transformer)
2. **Encoder-only** models (BERT family) for sentiment analysis, plus semantic search and clustering with sentence embeddings
3. **Decoder-only** (GPT-style) models for text generation, including a look at the next-token distribution that makes them autoregressive
4. **Running a GPT-type model locally with Ollama** and comparing its answers with a frontier model (Claude)

Everything runs locally on a MacBook (Apple M2, CPU/MPS). The `pip install` cell from the course notebook is not needed here, because the packages are installed in the environment.

In [1]:
import time, warnings
import numpy as np
import torch
import transformers
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, EncoderDecoderModel,
                          AutoModelForCausalLM, AutoModel, pipeline)
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()
torch.manual_seed(0)
print("transformers", transformers.__version__, "| torch", torch.__version__)

transformers 5.17.0 | torch 2.11.0


## 1. Encoder-decoder: machine translation

The encoder reads the whole source sentence with bidirectional self-attention. The decoder generates the target sentence one token at a time, using masked self-attention over what it has produced so far plus **cross-attention** to the encoder's outputs.

### 1a. `google/bert2bert_L-24_wmt_en_de` (English to German)

Code from the course notebook, unchanged except for a timer.

In [2]:
model_name = "google/bert2bert_L-24_wmt_en_de"
tokenizer = AutoTokenizer.from_pretrained(model_name, pad_token="<pad>", eos_token="</s>", bos_token="<s>")
model = EncoderDecoderModel.from_pretrained(model_name)

input_text = "Plants create energy through a process known as"
inputs = tokenizer(input_text, return_tensors="pt", add_special_tokens=False).input_ids
t0 = time.time()
outputs = model.generate(inputs)[0]
translated_text = tokenizer.decode(outputs, skip_special_tokens=True)
print("Translated text:", translated_text)
print(f"({time.time() - t0:.1f}s)")

Loading weights:   0%|          | 0/1019 [00:00<?, ?it/s]

Translated text: Pflanzen erzeugen Energie durch einen so genannten Prozess.
(9.0s)


In [3]:
# A few more sentences, including a complete one (the course example is an unfinished sentence).
for s in ["I love machine learning.",
          "The weather in Goa is very pleasant today.",
          "Plants create energy through a process known as photosynthesis."]:
    ids = tokenizer(s, return_tensors="pt", add_special_tokens=False).input_ids
    print(f"{s}\n  -> {tokenizer.decode(model.generate(ids)[0], skip_special_tokens=True)}")
print("\nparameters:", sum(p.numel() for p in model.parameters()) / 1e6, "M")
print("encoder:", type(model.encoder).__name__, "| decoder:", type(model.decoder).__name__,
      "| decoder has cross-attention:", model.decoder.config.add_cross_attention)

I love machine learning.
  -> Ich liebe maschinelles Lernen.


The weather in Goa is very pleasant today.
  -> das Wetter in Goa ist sehr angenehm heute.


Plants create energy through a process known as photosynthesis.
  -> Pflanzen erzeugen Energie durch einen Prozess, der Photosynthese genannt wird.

parameters: 771.947726 M
encoder: BertGenerationEncoder | decoder: BertGenerationDecoder | decoder has cross-attention: True


**Observations (bert2bert):** this model is a 772M-parameter encoder-decoder built from two BERT stacks. The decoder has `add_cross_attention = True`, which is the cross-attention link between the two halves. The course input is an *unfinished* sentence ("...through a process known as"), and the model produced *Pflanzen erzeugen Energie durch einen so genannten Prozess.* ("through a so-called process."). It quietly dropped the dangling "as" and added a full stop, because a translation model trained on complete sentence pairs always outputs a complete sentence. Given the full sentence, it gives an exact translation: *...durch einen Prozess, der Photosynthese genannt wird.* "I love machine learning" becomes *Ich liebe maschinelles Lernen*, and T5 produced the same German below. One small flaw: *das Wetter* is not capitalised at the start of the sentence.

### 1b. `google-t5/t5-base` (English to French, task given as a text prefix)

In [4]:
tokenizer = AutoTokenizer.from_pretrained("google-t5/t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google-t5/t5-base")

for s in ["Let's go have some wine, cause thats how we shine",
          "I love machine learning.",
          "The Transformer was introduced for machine translation."]:
    ids = tokenizer("translate English to French: " + s, return_tensors="pt")
    out = model.generate(**ids, max_length=40)
    print(f"{s}\n  -> {tokenizer.decode(out[0], skip_special_tokens=True)}")

# The same model does other tasks when the prefix changes.
for prefix, s in [("translate English to German: ", "I love machine learning."),
                  ("summarize: ", "The Transformer architecture was introduced in 2017 for machine translation. "
                   "It replaced recurrence with attention, which lets every token look at every other token "
                   "in a single step, and made training on large datasets much faster because it parallelises well.")]:
    ids = tokenizer(prefix + s, return_tensors="pt")
    out = model.generate(**ids, max_length=60)
    print(f"[{prefix.strip()}] -> {tokenizer.decode(out[0], skip_special_tokens=True)}")

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Let's go have some wine, cause thats how we shine
  -> Allons en déguster un peu de vin, car c'est ainsi que nous brillons


I love machine learning.
  -> J'aime l'apprentissage par machine.


The Transformer was introduced for machine translation.
  -> Le transformateur a été introduit pour la traduction automatique.


[translate English to German:] -> Ich liebe maschinelles Lernen.


[summarize:] -> the Transformer architecture was introduced in 2017 for machine translation . it replaced recurrence with attention, which lets every token look at every other token . it made training on large datasets much faster because it parallelises well .


**Observations (T5):** the translations are fluent and mostly correct. "The Transformer was introduced for machine translation" is rendered perfectly, including the technical term *traduction automatique*. For "I love machine learning" it produced *apprentissage par machine*, which is understandable but less idiomatic than the standard *apprentissage automatique*. The same term was handled well in one sentence and poorly in the other, depending on context. On the informal course sentence it handled the slang ("cause thats how we shine" became *car c'est ainsi que nous brillons*), but it turned "have some wine" into *déguster un peu de vin* ("taste a little wine"), which is a small shift in meaning.

The interesting architectural point is that **one** encoder-decoder model does several tasks: changing only the text prefix (`translate English to German:`, `summarize:`) switched it to German and to summarisation. T5 was trained as a single text-to-text model. The summary is lower-cased with spaces before full stops, an artefact of how T5's pre-training data was tokenised, and it mostly copies phrases from the input. That is extractive-style behaviour, typical of a base-size model.

### 1c. `rvv-karma/English2Hinglish-Flan-T5-Base` (English to Hinglish)

In [5]:
model_name = "rvv-karma/English2Hinglish-Flan-T5-Base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
for s in ["life is crazy. Not me", "I am going to the market to buy vegetables.", "What are you doing tonight?"]:
    ids = tokenizer(s, return_tensors="pt")
    out = model.generate(**ids, max_length=40)
    print(f"{s}\n  -> {tokenizer.decode(out[0], skip_special_tokens=True)}")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

life is crazy. Not me
  -> Life crazy haiNot me


I am going to the market to buy vegetables.
  -> Mai market jaa raha hoon toh vegetables kharidne ke liye.


What are you doing tonight?
  -> Aaj raat kya aa raha hai?


**Observations (Hinglish):** this is a small Flan-T5 fine-tuned for code-mixed output. "I am going to the market to buy vegetables" became *Mai market jaa raha hoon toh vegetables kharidne ke liye*, which is natural Hinglish (the *toh* is slightly odd). But the course example lost a space (*Life crazy haiNot me*), and "What are you doing tonight?" became *Aaj raat kya aa raha hai?* ("What is coming tonight?"), which is fluent but **wrong**. A fluent output is not the same as a correct one. A reader who doesn't know Hindi would not notice the error.

## 2. Encoder-only: understanding text

Encoder-only models (BERT and its descendants) use bidirectional self-attention. Every token sees the whole input, so they produce rich representations of existing text rather than generating new text.

### 2a. Sentiment analysis (course example)

In [6]:
classifier = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
print("model:", classifier.model.name_or_path)
sentences = [
    "I'd want to kick you and ensure that you are hurt",
    "This lab was surprisingly fun.",
    "The movie was not bad at all.",
    "The food was cold and the service was slow.",
    "It is a book.",
    "Oh great, another Monday.",
]
for s, r in zip(sentences, classifier(sentences)):
    print(f"{r['label']:8s} {r['score']:.4f}  {s}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

model: distilbert/distilbert-base-uncased-finetuned-sst-2-english


NEGATIVE 0.9742  I'd want to kick you and ensure that you are hurt
POSITIVE 0.9998  This lab was surprisingly fun.
POSITIVE 0.9980  The movie was not bad at all.
NEGATIVE 0.9998  The food was cold and the service was slow.
POSITIVE 0.9991  It is a book.
POSITIVE 0.9982  Oh great, another Monday.


**Observations (sentiment):** the encoder-only classifier (DistilBERT fine-tuned on SST-2 movie reviews) gets the clear cases right. That includes negation: "not bad at all" is correctly POSITIVE, which a bag-of-words model would likely get wrong. It shows that the encoder uses word order and context. But it has **no neutral class**. "It is a book." is labelled POSITIVE with 0.9991 confidence, because a two-way softmax must put its mass somewhere, and the score is not a calibrated certainty. It also misses sarcasm: "Oh great, another Monday." is POSITIVE with 0.998. The course example (a threat of violence) is correctly NEGATIVE, but sentiment is not the same as toxicity or safety. Using a sentiment model as a content filter would be a task mismatch.

### 2b. Semantic search and clustering (encoder-only use cases from the handout)

`sentence-transformers/all-MiniLM-L6-v2` is a small BERT-style encoder. Mean-pooling its token outputs gives one vector per sentence. Cosine similarity between vectors measures closeness of *meaning*, so a query can match a document that shares no words with it.

In [7]:
enc_name = "sentence-transformers/all-MiniLM-L6-v2"
enc_tok = AutoTokenizer.from_pretrained(enc_name)
encoder = AutoModel.from_pretrained(enc_name)

def embed(texts):
    batch = enc_tok(texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        out = encoder(**batch).last_hidden_state
    mask = batch["attention_mask"].unsqueeze(-1).float()
    vec = (out * mask).sum(1) / mask.sum(1)            # mean pooling over real tokens
    return torch.nn.functional.normalize(vec, dim=1)

docs = [
    "How do I reset my account password?",
    "Steps to recover a forgotten login credential",
    "Our office is closed on public holidays.",
    "Refunds are processed within 7 business days.",
    "What is the return policy for damaged items?",
    "The cafeteria serves lunch from 12 to 2 pm.",
    "Bayesian networks represent joint distributions compactly.",
    "Graphical models factorise probability distributions.",
]
D = embed(docs)
for q in ["I can't log in, forgot my password", "when will I get my money back", "probabilistic graphical models"]:
    scores = (embed([q]) @ D.T).squeeze(0)
    top = scores.argsort(descending=True)[:2]
    print(f"query: {q}")
    for i in top:
        print(f"   {scores[i]:.3f}  {docs[i]}")

# keyword baseline: word overlap would miss the second document for the first query
q = set("i can't log in, forgot my password".split())
print("\nshared words with doc 1:", q & set(docs[1].lower().split()))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

query: I can't log in, forgot my password
   0.757  How do I reset my account password?
   0.694  Steps to recover a forgotten login credential
query: when will I get my money back
   0.561  Refunds are processed within 7 business days.
   0.369  What is the return policy for damaged items?
query: probabilistic graphical models
   0.803  Graphical models factorise probability distributions.
   0.598  Bayesian networks represent joint distributions compactly.

shared words with doc 1: set()


In [8]:
# Clustering: group the documents by meaning with a tiny k-means on the embeddings.
def kmeans(X, k, iters=50, seed=0):
    g = torch.Generator().manual_seed(seed)
    C = X[torch.randperm(len(X), generator=g)[:k]].clone()
    for _ in range(iters):
        assign = (X @ C.T).argmax(1)
        C = torch.stack([torch.nn.functional.normalize(X[assign == j].mean(0), dim=0) if (assign == j).any() else C[j] for j in range(k)])
    return assign

assign = kmeans(D, 4)
for j in range(4):
    print(f"cluster {j}:", [docs[i] for i in range(len(docs)) if assign[i] == j])

cluster 0: ['Our office is closed on public holidays.', 'What is the return policy for damaged items?']
cluster 1: ['How do I reset my account password?', 'Steps to recover a forgotten login credential']
cluster 2: ['Bayesian networks represent joint distributions compactly.', 'Graphical models factorise probability distributions.']
cluster 3: ['Refunds are processed within 7 business days.', 'The cafeteria serves lunch from 12 to 2 pm.']


**Observations (semantic search and clustering):** for "I can't log in, forgot my password", the second result, "Steps to recover a forgotten login credential", shares **no words** with the query (the overlap printed above is the empty set), yet scores 0.694. A keyword search would miss it completely. "when will I get my money back" matched the refund policy, and "probabilistic graphical models" matched both BN sentences. This is what encoder-only models are used for in retrieval and RAG: they turn meaning into geometry.

The k-means clustering was only partly right. It correctly grouped the two password documents and the two graphical-model documents, but it paired "office closed on holidays" with "return policy", and "refunds" with "cafeteria lunch hours". With 8 short documents and k=4, the leftover administrative sentences are all roughly equally dissimilar, so the split among them is fairly arbitrary. The retrieval results are more trustworthy than the clusters.

## 3. Decoder-only: GPT-style generation

A decoder-only model is exactly the autoregressive model from the BN lab: it outputs P(x_t | x_1, ..., x_{t-1}) over the vocabulary, and text is produced by picking or sampling one token at a time. Masked (causal) self-attention ensures position t sees only earlier tokens.

### 3a. Course example

In [9]:
generator = pipeline(task="text-generation", model="gpt2")
torch.manual_seed(0)
output = generator("The future of AI is", max_length=30, num_return_sequences=1)
print(output[0]["generated_text"])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

The future of AI is uncertain. In fact, the AI community has been largely silent on the topic. In July, the IBM Research Group announced that they were working on a new AI-powered computer, called ARBOL. That is, they are hoping to create a machine that can "talk to, read and process the brain's signals." Although the technology wasn't officially announced at the time, the plan is to bring the machine to market in the next five years.

AI is currently in the "near future," but there are a number of technical challenges that need to be overcome in order to be able to make an "AI-powered machine." There are a number of things that need to be done right now to make things happen. In the meantime, it's still the "next generation" of AI, and while it is not perfect, it's still a great leap forward.

Now that we've covered this topic, I wanted to give a quick rundown of some of the issues that are being raised. First, is the future of AI a good idea? I think it is.

First, there are a number

### 3b. The next-token distribution, and greedy vs sampling

In [10]:
tok = AutoTokenizer.from_pretrained("gpt2")
lm = AutoModelForCausalLM.from_pretrained("gpt2")
lm.eval()

prompt = "The future of AI is"
ids = tok(prompt, return_tensors="pt").input_ids
with torch.no_grad():
    logits = lm(ids).logits[0, -1]
probs = torch.softmax(logits, dim=-1)
print(f"vocabulary size = {probs.numel()}, sum of probabilities = {probs.sum().item():.6f}")
top = probs.topk(8)
print(f"P(next token | '{prompt}'):")
for p, i in zip(top.values, top.indices):
    print(f"   {p.item():.4f}  {tok.decode(int(i))!r}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

vocabulary size = 50257, sum of probabilities = 1.000027
P(next token | 'The future of AI is'):
   0.0613  ' uncertain'
   0.0586  ' in'
   0.0451  ' not'
   0.0403  ' a'
   0.0364  ' still'
   0.0247  ' going'
   0.0190  ' very'
   0.0186  ' now'


In [11]:
def generate(prompt, greedy, n=3, max_new_tokens=25, seed=0):
    ids = tok(prompt, return_tensors="pt").input_ids
    outs = []
    for k in range(n):
        torch.manual_seed(seed + k)
        out = lm.generate(ids, max_new_tokens=max_new_tokens, do_sample=not greedy,
                          top_k=0 if not greedy else None, temperature=1.0 if not greedy else None,
                          pad_token_id=tok.eos_token_id)
        outs.append(tok.decode(out[0], skip_special_tokens=True))
    return outs

print("GREEDY (argmax at every step):")
for t in generate("The future of AI is", greedy=True):
    print("  ", t.replace("\n", " "))
print("\nSAMPLING (temperature 1, full distribution):")
for t in generate("The future of AI is", greedy=False):
    print("  ", t.replace("\n", " "))

GREEDY (argmax at every step):


   The future of AI is uncertain. The future of AI is uncertain.  The future of AI is uncertain. The future of AI is uncertain.
   The future of AI is uncertain. The future of AI is uncertain.  The future of AI is uncertain. The future of AI is uncertain.
   The future of AI is uncertain. The future of AI is uncertain.  The future of AI is uncertain. The future of AI is uncertain.

SAMPLING (temperature 1, full distribution):


   The future of AI is a gritty future with big stories and is hampered by rigid bipartisan power structures that discourage advance thinking. When people feel compelled to do
   The future of AI is up in the air, and perhaps the humans would want to continue doing exactly that. For now, Watson will be the main
   The future of AI is not tied to history or preferences or paradigm, but rather technology breaks it down into distinct little parts, the aspects of people who


In [12]:
# Masked self-attention check: the logits at position t must not depend on later tokens.
# Run in float64 so that genuine differences are not confused with float32 round-off.
lm64 = AutoModelForCausalLM.from_pretrained("gpt2", dtype=torch.float64).eval()
a = tok("The cat sat on the mat", return_tensors="pt").input_ids
b = tok("The cat sat on the sofa and slept", return_tensors="pt").input_ids
c = tok("A dog sat on the mat", return_tensors="pt").input_ids
n = 4  # shared prefix "The cat sat on"
with torch.no_grad():
    la, lb, lc = lm64(a).logits[0], lm64(b).logits[0], lm64(c).logits[0]
print("same prefix, different later tokens -> max |logit diff| on positions 0-3:", (la[:n] - lb[:n]).abs().max().item())
print("different earlier tokens            -> max |logit diff| on positions 2-5:", (la[2:6] - lc[2:6]).abs().max().item())

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

same prefix, different later tokens -> max |logit diff| on positions 0-3: 1.5631940186722204e-13
different earlier tokens            -> max |logit diff| on positions 2-5: 10.053798075903828


**Observations (decoder-only):**

1. **Next-token distribution:** GPT-2 outputs a probability for every one of its 50,257 tokens, and they sum to 1 (1.000027 in float32). After "The future of AI is", the most likely token is " uncertain", with only 6.1%. The distribution is very flat, so many continuations are plausible. This is the same object as P(X_t | X_1..X_{t-1}) in the BN lab. A neural network computes it instead of looking it up in a count table.
2. **Greedy decoding loops:** always taking the argmax gives "The future of AI is uncertain. The future of AI is uncertain. ...", and every run is identical. This is exactly the failure we saw with the greedy bigram model in Lab 7 ("the cat sat on the cat sat on ..."): a deterministic argmax walk falls into a cycle of high-probability tokens. **Sampling** gives three different, varied (and not always sensible) continuations. This is why practical decoders use sampling with temperature, top-k or top-p, or repetition penalties.
3. **Causal masking verified:** in float64, changing only the *later* tokens changes the logits at the earlier positions by at most 1.6e-13 (round-off), whereas changing an *earlier* token changes later logits by up to 10. Position t really does see only tokens 1..t. That is what makes the model autoregressive, and what allows a single forward pass to train on every next-token prediction in a sequence at once.
4. The course pipeline output (the long IBM/"ARBOL" paragraph) is fluent but invented: there is no such announcement. GPT-2 is a language model, not a knowledge base.

## 4. Running a GPT-type model locally with Ollama

`Run_Ollama.ipynb` runs Ollama on Colab through an xterm and queries `mistral` through LangChain. Here Ollama runs natively on the Mac, with `llama3.2:3b` (3B parameters, 2 GB quantised). The LangChain code is the same as in the course notebook.

In [13]:
import subprocess
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

NAME           ID              SIZE      MODIFIED     
llama3.2:3b    a80c4f17acd5    2.0 GB    8 months ago    



In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM

template = """Question: {question}

Answer: Let's think step by step."""
prompt = ChatPromptTemplate.from_template(template)
model = OllamaLLM(model="llama3.2:3b", temperature=0, seed=0)
chain = prompt | model

answers = {}
questions = [
    "Who is Sir Issac Newton",
    "A bat and a ball cost 1.10 dollars in total. The bat costs 1.00 dollar more than the ball. How much does the ball cost?",
    "How many times does the letter r appear in the word strawberry?",
    "Write a Python function that returns the n-th Fibonacci number iteratively.",
    "What is the capital of Australia?",
    "Explain in two sentences what cross-attention does in a Transformer.",
]
for q in questions:
    t0 = time.time()
    answers[q] = chain.invoke({"question": q})
    print("=" * 80)
    print("Q:", q, f"   ({time.time() - t0:.1f}s)")
    print("-" * 80)
    print(answers[q].strip())

Q: Who is Sir Issac Newton    (9.1s)
--------------------------------------------------------------------------------
Let's break down the question step by step to understand who Sir Isaac Newton is.

Step 1: Identify the context
The question mentions Sir Isaac Newton, which suggests that we're talking about a historical figure, likely a scientist or mathematician.

Step 2: Recall historical figures
We know that there have been many influential scientists and mathematicians throughout history. Some notable ones come to mind, such as Galileo Galilei, Albert Einstein, and Leonardo da Vinci.

Step 3: Connect to the question
The question specifically mentions Sir Isaac Newton. We need to find out who he was and what he's known for.

Step 4: Research and recall
Sir Isaac Newton was an English mathematician, physicist, and scientist who lived from 1643 to 1727. He's widely recognized as one of the most influential scientists in history.

Step 5: Summarize key facts
Newton made major contribu

Q: A bat and a ball cost 1.10 dollars in total. The bat costs 1.00 dollar more than the ball. How much does the ball cost?    (4.9s)
--------------------------------------------------------------------------------
To solve this problem, let's break it down step by step.

Let's say the cost of the ball is x dollars.

Since the bat costs 1.00 dollar more than the ball, the cost of the bat is x + 1.00 dollars.

The total cost of the bat and the ball together is 1.10 dollars. So, we can set up an equation:

x + (x + 1.00) = 1.10

Combine like terms:

2x + 1.00 = 1.10

Subtract 1.00 from both sides:

2x = 0.10

Divide both sides by 2:

x = 0.05

So, the ball costs 0.05 dollars, or 5 cents.


Q: How many times does the letter r appear in the word strawberry?    (3.0s)
--------------------------------------------------------------------------------
To find out how many times the letter "r" appears in the word "strawberry", let's break it down step by step:

1. Write down the word: strawberry
2. Look at each letter individually: s-t-r-a-w-b-e-r-r-y
3. Count the number of times the letter "r" appears: It appears twice.

So, the letter "r" appears twice in the word "strawberry".


Q: Write a Python function that returns the n-th Fibonacci number iteratively.    (3.1s)
--------------------------------------------------------------------------------
def fibonacci(n):
    if n <= 0:
        return "Input should be a positive integer."
    elif n == 1:
        return 0
    elif n == 2:
        return 1
    else:
        a, b = 0, 1
        for _ in range(n - 2):
            a, b = b, a + b
        return b


Q: What is the capital of Australia?    (8.2s)
--------------------------------------------------------------------------------
To find the answer, let's break it down step by step:

 Step 1: Identify the country in question - We are looking for the capital of Australia.

 Step 2: Recall general knowledge about Australia - Australia is a country located in the Southern Hemisphere, known for its unique wildlife and cultural diversity.

 Step 3: Consider common capitals of countries - Many countries have capitals that are well-known, such as Paris for France or Beijing for China.

 Step 4: Apply knowledge of Australian geography - Australia is a vast and sparsely populated country, with many major cities and regional centers.

 Step 5: Recall specific information about Australian cities - One of the largest and most well-known cities in Australia is Canberra.

 Step 6: Confirm Canberra as the capital - Canberra has been the capital of Australia since 1908, and it is home to many national

Q: Explain in two sentences what cross-attention does in a Transformer.    (2.8s)
--------------------------------------------------------------------------------
In a Transformer, cross-attention allows the model to attend to all positions in the input sequence simultaneously and weigh their importance, while also considering the context provided by the output sequence. This enables the model to capture long-range dependencies and relationships between different parts of the input and output sequences.


**Observations (Ollama, llama3.2:3b, temperature 0):** each answer took 3 to 27 s on the M2, entirely offline. The "Let's think step by step" template from the course notebook makes the model narrate steps even for trivial questions. The Newton and Canberra answers are long because the template forces a step-by-step format.

| question | local llama3.2:3b | correct? |
|---|---|---|
| Who is Isaac Newton | English mathematician and physicist, 1643 to 1727, laws of motion, gravitation, calculus, optics, *Principia* | yes (it also silently corrected the prompt's spelling "Issac") |
| bat and ball | sets up x + (x + 1.00) = 1.10 and gets **0.05** | yes, and it avoided the intuitive wrong answer 0.10 |
| r's in "strawberry" | spells out s-t-r-a-w-b-e-r-r-y, then says **"twice"** | **no**: the correct answer is 3, and its own spelling shows three r's |
| iterative Fibonacci | correct loop | works, but uses the 1-indexed convention fib(1) = 0 without saying so, and returns a *string* on invalid input instead of raising an error |
| capital of Australia | Canberra | yes, but the reasoning is wrong: it calls Canberra "one of the largest" Australian cities (it is not), and "capital since 1908" is imprecise (the site was chosen in 1908, and Parliament moved there in 1927) |
| cross-attention | "attend to all positions in the input sequence... considering the context provided by the output sequence" | vague. It never says that queries come from the decoder and keys/values from the encoder, and the "long-range dependencies" sentence describes self-attention in general |

### Comparison with a frontier model

The same questions answered by **Claude (Opus 5.5)**, the model used as the coding assistant for these labs:

| question | Claude | local 3B model |
|---|---|---|
| Newton | English mathematician, physicist and astronomer (1643 to 1727); laws of motion and universal gravitation (*Principia*, 1687), co-inventor of calculus with Leibniz, work on optics and the reflecting telescope | same core facts, much more padding |
| bat and ball | 5 cents (ball 0.05, bat 1.05) | same, correct |
| r's in strawberry | **3** (st-**r**-awbe-**rr**-y) | **2, wrong** |
| Fibonacci | `a, b = 0, 1; for _ in range(n): a, b = b, a + b; return a`, with F(0) = 0 and F(1) = 1, raising `ValueError` for negative n | correct but uses a non-standard, undocumented indexing |
| capital of Australia | Canberra, a purpose-built capital. The site was chosen in 1908 as a compromise between Sydney and Melbourne, and Parliament moved there in 1927 | right answer, wrong supporting facts |
| cross-attention | In a decoder layer, the queries come from the decoder's own states, while the keys and values come from the encoder's output. So each target position can pull in the source information it needs (for example, which source word to translate next) | vague and partly off-topic |

**What the comparison shows:**
- On knowledge and easy reasoning (Newton, bat and ball, Canberra), the 3B local model gets the **headline answer right**, but its supporting details are less reliable.
- It fails a task that needs attention to characters rather than tokens (counting r's). Its step-by-step spelling even contains the right answer, and it still reported the wrong count, so the displayed "reasoning" did not determine the answer.
- Explanations of technical concepts are where the size gap shows most. The small model produces text that sounds right but lacks the precise mechanism.
- Trade-offs: the local model is free, private (nothing leaves the laptop), works offline, and takes a few seconds per answer on a laptop CPU/GPU. The frontier model is far more accurate and precise, but needs a network connection and an external service.

## Summary

| architecture | attention | typical use | model used here | what we observed |
|---|---|---|---|---|
| encoder-decoder | bidirectional self-attention in the encoder, masked self-attention + cross-attention in the decoder | translation, summarisation (sequence to sequence) | bert2bert (EN to DE), T5-base (EN to FR/DE, summarise), Flan-T5 (EN to Hinglish) | good translations from T5 and bert2bert. One T5 model does many tasks through text prefixes. The small Hinglish model is fluent but sometimes wrong in meaning |
| encoder-only | bidirectional self-attention | classification, semantic search, clustering, retrieval | DistilBERT (sentiment), MiniLM (embeddings) | handles negation, but has no neutral class, is overconfident, and misses sarcasm. Semantic search finds matches with zero word overlap |
| decoder-only | masked (causal) self-attention | generation, chat, code | GPT-2, llama3.2:3b via Ollama | explicit next-token distribution. Greedy decoding loops just like the bigram model, and sampling adds variety. The local 3B model is useful but makes factual and character-level errors that a frontier model does not |

The common thread with the rest of the course: a decoder-only LM is the autoregressive factorisation P(x_1..x_T) = product of P(x_t | x_<t) from the Bayesian-network lab, with the conditional table replaced by a Transformer, and the causal mask is what enforces that factorisation.